# FE BAR Step 4 - Lakebase scores read path validation

Validates the SNAPSHOT synced table `fe_bar_lakebase.fraud.txn_risk_current` (source `workshop_guy_catalog.fe_bar_a0.scored_transactions`, PK `txn_id`, database `fe_bar_db`):

* Synced table status and pipeline
* Row count match UC vs Postgres
* Traced seeded-cohort txn returns the same `risk_score` and `recommended_action` from Postgres
* p50/p95/p99 latency over 200 PK lookups on one reused OAuth connection (same pattern as `lakebase_cases.py`)

In [0]:
import time, json, os, sys
# Notebook lives in validation_notebooks/; repo modules are one level up
sys.path.insert(0, os.path.dirname(os.getcwd()))
import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient
from databricks.sdk import WorkspaceClient
from lakebase_cases import LakebaseCases  # repo module: OAuth token via w.postgres.generate_database_credential

SCORED_TABLE = "workshop_guy_catalog.fe_bar_a0.scored_transactions"
FEATURE_TABLE = "workshop_guy_catalog.fe_bar_a0.txn_features_gold"
SYNCED_TABLE = "fe_bar_lakebase.fraud.txn_risk_current"
PG_TABLE = "fraud.txn_risk_current"
MODEL_NAME = "workshop_guy_catalog.fe_bar_a0.fraud_risk_model"

w = WorkspaceClient()
st = w.postgres.get_synced_table(name=f"synced_tables/{SYNCED_TABLE}")
print("synced table:", SYNCED_TABLE)
print("  detailed_state:", st.status.detailed_state)
print("  pipeline_id:   ", st.status.pipeline_id)
print("  last_sync:     ", st.status.last_sync)
ep = w.postgres.get_endpoint(name="projects/fe-bar-fraud-analytics/branches/production/endpoints/primary")
print("endpoint:", ep.status.current_state, "| suspend_timeout:", ep.status.suspend_timeout_duration,
      "| CU min/max:", ep.status.autoscaling_limit_min_cu, ep.status.autoscaling_limit_max_cu)

synced table: fe_bar_lakebase.fraud.txn_risk_current
  detailed_state: SyncedTableState.SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE
  pipeline_id:    f8690359-4bb1-45b9-8fdb-5bd892239db2
  last_sync:      SyncedTablePosition(delta_table_sync_info=DeltaTableSyncInfo(delta_commit_time=seconds: 1792801405
, delta_commit_version=9), sync_end_time=seconds: 1792801590
nanos: 446094000
, sync_start_time=seconds: 1792801569
nanos: 503792000
)
endpoint: EndpointStatusState.ACTIVE | suspend_timeout: seconds: 300
 | CU min/max: 1.0 1.0


In [0]:
lb = LakebaseCases()
conn = lb._get_conn()  # one reused connection for every check below
with conn.cursor() as cur:
    cur.execute("""SELECT column_name, data_type FROM information_schema.columns
                   WHERE table_schema='fraud' AND table_name='txn_risk_current' ORDER BY ordinal_position""")
    print(pd.DataFrame(cur.fetchall(), columns=["column", "pg_type"]).to_string(index=False))
    cur.execute("""SELECT a.attname FROM pg_index i JOIN pg_attribute a ON a.attrelid=i.indrelid AND a.attnum = ANY(i.indkey)
                   WHERE i.indrelid='fraud.txn_risk_current'::regclass AND i.indisprimary""")
    print("\nPostgres primary key:", [r[0] for r in cur.fetchall()])
conn.commit()

            column                  pg_type
            txn_id                     text
       customer_id                     text
            txn_ts timestamp with time zone
            amount         double precision
        risk_score         double precision
recommended_action                     text
  top_shap_signals                     text
     model_version                     text
         scored_at timestamp with time zone

Postgres primary key: ['txn_id']


In [0]:
uc_rows = spark.table(SCORED_TABLE).count()
uc_scored_at = spark.table(SCORED_TABLE).agg({"scored_at": "max"}).collect()[0][0]
with conn.cursor() as cur:
    cur.execute(f"SELECT count(*), max(scored_at), min(model_version), max(model_version) FROM {PG_TABLE}")
    pg_rows, pg_scored_at, pg_mv_min, pg_mv_max = cur.fetchone()
conn.commit()
print(pd.DataFrame([
    {"store": "UC scored_transactions", "rows": uc_rows, "max_scored_at": str(uc_scored_at)},
    {"store": "Postgres fraud.txn_risk_current", "rows": pg_rows, "max_scored_at": str(pg_scored_at)},
]).to_string(index=False))
print(f"\nrow count match: {uc_rows == pg_rows} | model_version in PG: {pg_mv_min}..{pg_mv_max}")
assert uc_rows == pg_rows

                          store   rows                    max_scored_at
         UC scored_transactions 464632       2026-10-24 00:22:56.349759
Postgres fraud.txn_risk_current 464632 2026-10-24 00:22:56.349759+00:00

row count match: True | model_version in PG: 1..1


In [0]:
# Same deterministic rule as fraud_batch_scoring (cohort cutoff from the champion run's params)
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient(registry_uri="databricks-uc")
mv = client.get_model_version_by_alias(MODEL_NAME, "champion")
COHORT_P90 = float(client.get_run(mv.run_id).data.params["cohort_amt_ratio_p90"])
uc = spark.sql(f"""
  SELECT s.txn_id, s.risk_score, s.recommended_action, s.top_shap_signals, s.model_version, s.scored_at
  FROM {SCORED_TABLE} s JOIN {FEATURE_TABLE} g USING (txn_id)
  WHERE g.amt_to_monthly_spend >= {COHORT_P90} AND g.is_cross_border = 1 AND g.is_new_device = 1
    AND s.recommended_action = 'block' AND g.is_fraud
  ORDER BY s.risk_score DESC, s.txn_id LIMIT 1""").collect()[0].asDict()
TRACED_TXN_ID = uc["txn_id"]

with conn.cursor() as cur:
    cur.execute(f"SELECT txn_id, risk_score, recommended_action, top_shap_signals, model_version, scored_at "
                f"FROM {PG_TABLE} WHERE txn_id = %s", (TRACED_TXN_ID,))
    r = cur.fetchone()
conn.commit()
pg = dict(zip(["txn_id", "risk_score", "recommended_action", "top_shap_signals", "model_version", "scored_at"], r))

print("TRACED TXN_ID:", TRACED_TXN_ID)
cmp = pd.DataFrame([{"field": k, "uc": str(uc[k]), "postgres": str(pg[k]), "match": str(uc[k]) == str(pg[k])}
                    for k in ["risk_score", "recommended_action", "model_version"]])
print(cmp.to_string(index=False))
print("\ntop_shap_signals (postgres):", pg["top_shap_signals"])
assert uc["risk_score"] == pg["risk_score"] and uc["recommended_action"] == pg["recommended_action"]
print("\nTraced txn matches between UC and Postgres.")

TRACED TXN_ID: TXN-0234209
             field                 uc           postgres  match
        risk_score 0.9844108757126732 0.9844108757126732   True
recommended_action              block              block   True
     model_version                  1                  1   True

top_shap_signals (postgres): [{"feature": "is_new_device", "value": 1.0, "shap": 2.5724}, {"feature": "geo_distance_from_home", "value": 450.63, "shap": 1.8741}, {"feature": "is_online", "value": 1.0, "shap": 1.2001}]

Traced txn matches between UC and Postgres.


In [0]:
ids = [r.txn_id for r in spark.table(SCORED_TABLE).sample(fraction=0.002, seed=7).limit(200).collect()]
ids = (ids + [TRACED_TXN_ID] * 200)[:200]
sql = f"SELECT txn_id, risk_score, recommended_action, top_shap_signals FROM {PG_TABLE} WHERE txn_id = %s"

with conn.cursor() as cur:  # warm-up (wakes compute, primes plan cache); excluded from timings
    for t in ids[:10]:
        cur.execute(sql, (t,)); cur.fetchone()
conn.commit()

lat_ms, hits = [], 0
with conn.cursor() as cur:
    for t in ids:
        t0 = time.perf_counter()
        cur.execute(sql, (t,))
        row = cur.fetchone()
        lat_ms.append((time.perf_counter() - t0) * 1000)
        hits += row is not None
conn.commit()
lat = np.array(lat_ms)
print(f"lookups: {len(lat)} | hits: {hits} | single reused connection, OAuth token")
print(pd.DataFrame({"metric": ["p50", "p95", "p99", "mean", "max"],
                    "latency_ms": [np.percentile(lat, 50), np.percentile(lat, 95), np.percentile(lat, 99), lat.mean(), lat.max()]})
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))
lb.close()

lookups: 200 | hits: 200 | single reused connection, OAuth token
metric  latency_ms
   p50        2.53
   p95        3.71
   p99        4.76
  mean        2.71
   max        5.02


In [0]:
%pip install -q "psycopg[binary]==3.3.6" "databricks-sdk>=0.118.0"
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
